# ✈️ Flight Search — Kayak Crawl Tool

Este notebook muestra 3 formas de buscar vuelos con la herramienta `crawl_kayak_flights`:

1. **Solo la herramienta** — llamada directa, sin IA, sin CrewAI
2. **OpenAI + tool calling** — sin CrewAI, la IA decide cuándo usar la herramienta
3. **CrewAI single agent** — un mini-crew de 1 agente

**¿Necesitas CrewAI?** No para un solo agente. CrewAI añade valor cuando:
- Tienes múltiples agentes que se pasan contexto entre sí
- Necesitas HITL (human-in-the-loop) entre pasos
- Quieres memoria cross-session
- Necesitas ejecución paralela de tareas

## Setup

In [ ]:
import sys, os, json
from pathlib import Path
from dotenv import load_dotenv

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "tests" else Path.cwd()
load_dotenv(PROJECT_DIR / ".env")

REPO_ROOT = str(PROJECT_DIR.parent.parent)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("OPENAI_API_KEY:", "✓" if os.getenv("OPENAI_API_KEY") else "✗")

---
## 1. Solo la herramienta (sin IA, sin CrewAI)

La forma más simple. Llamas a la función directamente y obtienes el HTML/Markdown de Kayak.

**Pros:** Rápido, sin costes de API de LLM, datos crudos.

**Contras:** Tú tienes que parsear el resultado.

In [ ]:
from src.lab4_tip_planner.tools import crawl_kayak_flights

# Llamada directa a la herramienta
raw_result = crawl_kayak_flights.run(
    departure="BCN",
    destination="KEF",
    date="2026-09-07",
    return_date="2026-09-14",
    adults=2,
)

data = json.loads(raw_result)
print(f"URL: {data['url']}")
print(f"Contenido ({len(data['content'])} chars):")
print(data['content'][:1000])

---
## 2. OpenAI + Tool Calling (sin CrewAI)

La IA decide cuándo llamar a la herramienta y parsea el resultado.
No necesitas CrewAI — solo la API de OpenAI con function calling.

**Pros:** Control total, sin overhead de CrewAI, la IA interpreta los resultados.

**Contras:** Más código manual para el loop de tool calling.

In [ ]:
from openai import OpenAI

client = OpenAI()

# Definir la herramienta como función de OpenAI
tools = [{
    "type": "function",
    "function": {
        "name": "crawl_kayak_flights",
        "description": "Busca vuelos reales haciendo crawling de Kayak con browser headless",
        "parameters": {
            "type": "object",
            "properties": {
                "departure": {"type": "string", "description": "Código IATA de salida (ej: BCN)"},
                "destination": {"type": "string", "description": "Código IATA de destino (ej: KEF)"},
                "date": {"type": "string", "description": "Fecha ida YYYY-MM-DD"},
                "return_date": {"type": "string", "description": "Fecha vuelta YYYY-MM-DD"},
                "adults": {"type": "integer", "description": "Número de adultos"},
            },
            "required": ["departure", "destination", "date"]
        }
    }
}]

# Pedir a la IA que busque vuelos
messages = [{
    "role": "user",
    "content": (
        "Busca vuelos directos de Barcelona a Reykjavik, ida 7 septiembre 2026, "
        "vuelta 14 septiembre, 2 adultos. Dame las 3 mejores opciones con precios reales."
    )
}]

print("Enviando petición a OpenAI...")
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    tool_choice="auto",
)

msg = response.choices[0].message
print(f"La IA quiere llamar a: {msg.tool_calls[0].function.name if msg.tool_calls else 'ninguna herramienta'}")

In [ ]:
# Ejecutar la herramienta que la IA pidió
if msg.tool_calls:
    tool_call = msg.tool_calls[0]
    args = json.loads(tool_call.function.arguments)
    print(f"Argumentos: {args}")
    
    # Ejecutar crawl real
    tool_result = crawl_kayak_flights.run(**args)
    print(f"\nKayak devolvió {len(tool_result)} chars")
    
    # Devolver resultado a la IA para que lo interprete
    messages.append(msg.model_dump())
    messages.append({
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": tool_result,
    })
    
    print("\nPidiendo a la IA que interprete los resultados...")
    final_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
    )
    
    print("\n" + "="*60)
    print("RESULTADO (OpenAI + Kayak, sin CrewAI):")
    print("="*60)
    print(final_response.choices[0].message.content)

---
## 3. CrewAI Single Agent

Un mini-crew con 1 agente. CrewAI gestiona el loop de herramientas automáticamente.

**Pros:** Más simple de escribir, retry automático, verbose logging.

**Contras:** Overhead de CrewAI, más tokens consumidos por los prompts internos.

In [ ]:
from crewai import Agent, Crew, Process, Task

flight_agent = Agent(
    role="Especialista en Vuelos",
    goal="Buscar vuelos reales en Kayak usando SOLO datos de la herramienta de crawling",
    backstory="Agente de viajes que solo trabaja con precios verificables de Kayak.",
    tools=[crawl_kayak_flights],
    verbose=True,
    allow_delegation=False,
)

flight_task = Task(
    description=(
        "Busca vuelos de Barcelona (BCN) a Reykjavik (KEF).\n"
        "Ida: 2026-09-07, Vuelta: 2026-09-14, 2 adultos.\n"
        "Solo vuelos directos.\n\n"
        "Usa la herramienta 'Kayak flight crawl' y propón las 3 mejores opciones "
        "con los precios REALES que aparecen en Kayak."
    ),
    expected_output="Las 3 mejores opciones de vuelo con precios reales de Kayak",
    agent=flight_agent,
)

crew = Crew(
    agents=[flight_agent],
    tasks=[flight_task],
    process=Process.sequential,
    verbose=True,
)

print("Ejecutando agente de vuelos con CrewAI...")
result = crew.kickoff()

print("\n" + "="*60)
print("RESULTADO (CrewAI + Kayak):")
print("="*60)
print(result.raw)

---
## Comparación

| Enfoque | CrewAI necesario | Coste LLM | Complejidad | Mejor para |
|---------|:---:|:---:|:---:|---|
| **1. Solo herramienta** | No | 0 | Bajo | Scripts, automatización |
| **2. OpenAI + tool calling** | No | Bajo (2 calls) | Medio | App custom, control total |
| **3. CrewAI single agent** | Sí | Alto (prompts internos) | Bajo (código) | Prototipado rápido, integración al crew |

**Recomendación:**
- Para **buscar vuelos y ya**: opción 1 o 2
- Para **integrar en el crew completo** con transporte, actividades y alojamiento: opción 3